# Lab 5: Vector Search in MongoDB and a Guardrail for Agents

This laboratory continues Lab 4 and covers **Part III of Course 4** (*Vector Search*): exact and approximate nearest-neighbor search (ENN, ANN), the HNSW index maintained by `mongot`, the stage `$vectorSearch`, pre- and post-filtering, the tuning of `numCandidates` by measuring recall, a retrieval pipeline for RAG, quantization and hybrid search with Reciprocal Rank Fusion. Part II returns to the agents of Labs 2 and 3: a tool that inspects the plan of a model-written pipeline with `explain()` *before* executing it.

**Prerequisites.** An Atlas cluster with the sample data and a read-write user: **Section 0 of Lab 4** describes how to create them. The laboratory does not depend on the code of Labs 3 and 4:

- the embeddings of the real data are generated in this notebook with `sentence-transformers` (model `all-MiniLM-L6-v2`, on CPU), unless the index of Lab 3 (`vector_index`, built with `nomic-embed-text` in `Airbnb_Embeddings.ipynb`) already exists, in which case it is reused;
- the helper functions of Lab 4 (`explain`, `plan_summary`, ...) are defined again below;
- the cells that call a model through Ollama (Lab 3) are marked *Optional*.

### Datasets

| Dataset | Where | Used for |
|---|---|---|
| `articles`, `authors` | `bdnv_lab` | The 8 documents of the Course 4 handout with 4-dimensional embeddings: every score can be verified by hand |
| `listings_emb` (or `sample_airbnb.listingsAndReviews` with the embeddings of Lab 3) | `bdnv_lab` | Vector search on real data: 2,000 listing summaries embedded with `all-MiniLM-L6-v2` |
| `articles_big` | `bdnv_lab` (generated in Part II) | The guardrail of Part II |

### Suggested schedule (2 hours)

| Section | Time | |
|---|---|---|
| Setting up the environment | 5 min | |
| The course dataset and the brute-force search | 10 min | |
| Part I: Vector search: concepts, the index, `$vectorSearch`, filters | 25 min | |
| Real data: embeddings (built in about 3 minutes if missing), recall@k | 20 min | |
| RAG retrieval, quantization, hybrid search | 20 min | |
| Exercises 1 and 3 (hand computations from the handout) | 25 min | in the lab |
| Part II: `explain()` as a guardrail | 15 min | |
| Exercises 2 and 4 | | **homework** |

### Limits of the free tier (M0)

An M0 cluster accepts at most **3** Atlas Search / Vector Search indexes. This laboratory creates at most two (`articles_vec` and, if the index of Lab 3 is missing, `listings_vec`). A student who completed Lab 3 already has two indexes (`vector_index` on `listingsAndReviews` and the `fullplot` index on `movies`); the existing index `vector_index` is then reused, and only `articles_vec` is added. The optional text index of the hybrid-search section needs a fourth slot. The final cell removes the indexes created here.

## Setting up the environment

### Installing the dependencies

`pymongo` is the MongoDB driver; `sentence-transformers` computes the embeddings (it is preinstalled on Colab); `langchain` is needed in Part II; `langchain-ollama` only in the optional cells; `python-dotenv` reads a local `.env` file.

In [ ]:
!pip install -q pymongo sentence-transformers langchain langchain-ollama python-dotenv

### Connecting to the cluster

The connection string is read from the Colab secret `MONGODB_URI`, from a `.env` file or from the environment (Lab 4, Section 0, step 7). The server version is printed because `$rankFusion` requires MongoDB 8.0 or later.

In [ ]:
import os
from getpass import getpass

from pymongo import MongoClient
from pymongo.errors import OperationFailure


def get_secret(name: str) -> str | None:
    # Read a secret from Colab, from a local .env file or from the environment.
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    return os.environ.get(name)


MONGODB_URI = get_secret("MONGODB_URI") or getpass("MongoDB Atlas connection string (read-write): ")

client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=15_000)
client.admin.command("ping")
print("Connected. Server version:", client.server_info()["version"])

airbnb = client["sample_airbnb"]["listingsAndReviews"]   # read only
mflix = client["sample_mflix"]                            # read only
lab = client["bdnv_lab"]                                  # scratch database: everything these labs create

### Checking the environment

The next cell verifies the prerequisites and reports what is missing, with a pointer to the step of Section 0 of Lab 4 that fixes it. Run it again after fixing a problem.

In [ ]:
def check(label: str, ok: bool, fix: str = "") -> bool:
    print(f"[{'OK' if ok else '!!'}] {label}" + ("" if ok else f"\n       -> {fix}"))
    return ok


try:
    databases = client.list_database_names()
except OperationFailure:
    databases = [name for name in ("sample_airbnb", "sample_mflix") if client[name].list_collection_names()]

HAS_AIRBNB = check("sample_airbnb is loaded", "sample_airbnb" in databases,
                   "Load the sample dataset (Lab 4, Section 0, step 5).")
HAS_MFLIX = check("sample_mflix is loaded", "sample_mflix" in databases,
                  "Load the sample dataset (Lab 4, Section 0, step 5).")
try:
    lab["_write_check"].insert_one({"ok": True})
    lab["_write_check"].drop()
    CAN_WRITE = True
except OperationFailure:
    CAN_WRITE = False
check("the user can write (scratch database bdnv_lab)", CAN_WRITE,
      "Use a user with the role 'Read and write to any database' (Lab 4, Section 0, step 3).")

try:
    LAB3_SEARCH_INDEXES = [index["name"] for index in airbnb.list_search_indexes()]
    HAS_SEARCH = True
except OperationFailure:
    LAB3_SEARCH_INDEXES, HAS_SEARCH = [], False
check("Atlas Search / Vector Search is available", HAS_SEARCH,
      "Vector search requires an Atlas cluster (or a local Atlas deployment).")
if HAS_SEARCH:
    print(f"[--] vector index of Lab 3 on listingsAndReviews: "
          f"{'present' if 'vector_index' in LAB3_SEARCH_INDEXES else 'absent (not needed)'}")

### Helper functions

| Function | Purpose |
|---|---|
| `show(documents)` | Prints documents as compact JSON (`Decimal128`, `ObjectId` and dates as strings) |
| `stage_counts(collection, pipeline)` | Prints the number of documents that leave each stage of a pipeline |
| `explain(collection, pipeline, ...)` | Runs the `explain` command for a pipeline or a `find` |
| `plan_summary(explanation)` | Summarizes a plan: stages, indexes, `nReturned`, `totalKeysExamined`, `totalDocsExamined`, time (see Lab 4, Part II) |
| `timed(function)` | Median wall-clock time of a call in milliseconds (including the network round trip) |
| `search_indexes`, `ensure_search_index` | List search indexes; create a search index if it is missing and wait until it is `READY` |

In [ ]:
from pymongo.operations import SearchIndexModel
import json
import time
from statistics import median


def show(documents, limit: int = 10) -> None:
    # Print documents as compact JSON.
    for i, document in enumerate(documents):
        if i == limit:
            print("...")
            break
        print(json.dumps(document, default=str, ensure_ascii=False))


def stage_counts(collection, pipeline: list[dict]) -> None:
    # Print the number of documents that leave each stage of a pipeline.
    for i in range(1, len(pipeline) + 1):
        result = list(collection.aggregate(pipeline[:i] + [{"$count": "n"}]))
        n = result[0]["n"] if result else 0
        print(f"({i}) {next(iter(pipeline[i - 1])):<18} -> {n} documents")


def timed(function, repeat: int = 3) -> float:
    # Median wall-clock time of function() in milliseconds.
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        function()
        times.append((time.perf_counter() - start) * 1000)
    return round(median(times), 1)


def explain(collection, pipeline: list[dict] | None = None, *, filter: dict | None = None,
            projection: dict | None = None, sort: dict | None = None, limit: int | None = None,
            hint=None, verbosity: str = "executionStats") -> dict:
    # Run the explain command for an aggregation pipeline or for a find().
    if pipeline is not None:
        command = {"aggregate": collection.name, "pipeline": pipeline, "cursor": {}}
    else:
        command = {"find": collection.name, "filter": filter or {}}
        if projection:
            command["projection"] = projection
        if sort:
            command["sort"] = sort
        if limit:
            command["limit"] = limit
    if hint:
        command["hint"] = hint
    return collection.database.command({"explain": command, "verbosity": verbosity})


def _walk_plan(node, stages: list, indexes: list) -> None:
    # Collect the stage names of a plan tree (root first) and the indexes it uses.
    if not isinstance(node, dict):
        return
    node = node.get("queryPlan", node)
    if isinstance(node.get("stage"), str):
        name = node["stage"]
        if "strategy" in node:                       # $lookup in SBE: EQ_LOOKUP(strategy)
            name += f"({node['strategy']})"
        stages.append(name)
        if "indexName" in node:
            indexes.append(node["indexName"])
    for key in ("inputStage", "outerStage", "innerStage"):
        _walk_plan(node.get(key), stages, indexes)
    for child in node.get("inputStages", []):
        _walk_plan(child, stages, indexes)


def _find_values(obj, key: str):
    # Yield every value stored under `key` anywhere in a nested document.
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == key:
                yield v
            yield from _find_values(v, key)
    elif isinstance(obj, list):
        for item in obj:
            yield from _find_values(item, key)


def plan_summary(explanation: dict, quiet: bool = False) -> dict:
    # Summarize the output of explain(): plan stages, indexes, metrics, remaining pipeline stages.
    stages = explanation.get("stages") or []
    has_cursor = bool(stages) and "$cursor" in stages[0]
    layer = stages[0]["$cursor"] if has_cursor else explanation
    plan, indexes = [], []
    _walk_plan(layer.get("queryPlanner", {}).get("winningPlan", {}), plan, indexes)
    stats = layer.get("executionStats", {})
    remaining = stages[1:] if has_cursor else stages
    pipeline_stages = []
    for stage in remaining:
        name = next(k for k in stage if k.startswith("$"))
        details = [f"out={stage['nReturned']}"] if "nReturned" in stage else []
        for key in ("strategy", "collectionScans", "indexesUsed", "totalDocsExamined", "usedDisk"):
            if key in stage and (key != "usedDisk" or stage[key]):
                details.append(f"{key}={stage[key]}")
        pipeline_stages.append(f"{name}({', '.join(details)})" if details else name)
    summary = {
        "plan": " <- ".join(plan) or "-",
        "indexes": indexes,
        "nReturned": stats.get("nReturned"),
        "keysExamined": stats.get("totalKeysExamined"),
        "docsExamined": stats.get("totalDocsExamined"),
        "timeMs": stats.get("executionTimeMillis"),
        "pipeline": pipeline_stages,
        "usedDisk": any(v is True for v in _find_values(explanation, "usedDisk")),
    }
    if not quiet:
        print(f"plan         : {summary['plan']}")
        print(f"indexes      : {summary['indexes'] or 'none'}")
        if stats:
            print(f"nReturned    : {summary['nReturned']} | keysExamined: {summary['keysExamined']} "
                  f"| docsExamined: {summary['docsExamined']} | executionTimeMillis: {summary['timeMs']}")
        if pipeline_stages:
            print(f"after query  : {' -> '.join(pipeline_stages)}")
        if summary["usedDisk"]:
            print("usedDisk     : True (a blocking stage spilled to disk)")
    return summary


def summary_table(rows: dict[str, dict]) -> None:
    # Print several plan summaries side by side.
    print(f"{'variant':<28}{'plan':<44}{'nReturned':>10}{'keys':>9}{'docs':>9}{'ms':>6}")
    for label, s in rows.items():
        print(f"{label:<28}{s['plan'][:43]:<44}{str(s['nReturned']):>10}{str(s['keysExamined']):>9}"
              f"{str(s['docsExamined']):>9}{str(s['timeMs']):>6}")


def search_indexes(collection) -> list[dict]:
    try:
        return list(collection.list_search_indexes())
    except OperationFailure as error:
        print(f"Cannot list search indexes on {collection.full_name}: {error.details.get('errmsg', error)}")
        return []


def wait_for_search_index(collection, name: str, timeout: int = 300) -> dict:
    # Wait until a search index is READY and queryable.
    start = time.time()
    while time.time() - start < timeout:
        index = next(iter(collection.list_search_indexes(name)), None)
        if index and index.get("queryable") and index.get("status") == "READY":
            print(f"Search index '{name}' is READY ({time.time() - start:.0f} s)")
            return index
        time.sleep(5)
    raise TimeoutError(f"Search index '{name}' is not ready after {timeout} s")


def ensure_search_index(collection, name: str, definition: dict, kind: str = "vectorSearch") -> dict:
    # Create a search index if it does not exist, then wait for it.
    if any(index["name"] == name for index in search_indexes(collection)):
        print(f"Search index '{name}' already exists")
    else:
        collection.create_search_index(SearchIndexModel(definition=definition, name=name, type=kind))
        print(f"Search index '{name}' created, building...")
    return wait_for_search_index(collection, name)

### The course dataset: `articles` and `authors`

The collection `articles` contains the eight documents of the Course 4 handout. The embeddings have only **four dimensions**, so that dot products can be computed by hand, and all vectors are **L2-normalized** ($\lVert e \rVert = 1$), as in Step 7 of Course 2. A real model such as `all-MiniLM-L6-v2` produces vectors with $d = 384$, and `nomic-embed-text` (Lab 3) vectors with $d = 768$. Each article also references an author in the collection `authors` (field `authorId`), as in the slides on `$lookup`. The query vector, obtained "with the same model", is $q = [0.5, 0.7, 0.5, 0.1]$.

| `_id` | `title` | `lang` | `year` | `tags` | `embedding` |
|---|---|---|---|---|---|
| A1 | Introducere în MongoDB | ro | 2023 | mongodb, nosql | [1.0, 0.0, 0.0, 0.0] |
| A2 | Aggregation pipeline pas cu pas | ro | 2024 | mongodb, pipeline | [0.6, 0.8, 0.0, 0.0] |
| A3 | Indecși B-tree și regula ESR | en | 2024 | index, mongodb | [0.5, 0.5, 0.5, 0.5] |
| A4 | Embeddings pentru text | ro | 2025 | nlp, vector | [0.0, 0.6, 0.8, 0.0] |
| A5 | HNSW explicat | en | 2025 | vector, index | [0.0, 0.0, 0.6, 0.8] |
| A6 | Tokenizare BPE | ro | 2025 | nlp | [0.0, 0.0, 0.0, 1.0] |
| A7 | Căutare hibridă | ro | 2026 | vector, search | [0.0, 0.8, 0.0, 0.6] |
| A8 | Quantization pentru vectori | en | 2026 | vector | [0.8, 0.0, 0.6, 0.0] |

The document also stores the name of the embedding model (`emb_model`), as recommended in the course: a query must be embedded with the same model as the stored documents.

In [ ]:
import math

ARTICLES = [
    ("A1", "Introducere în MongoDB",          "ro", 2023, ["mongodb", "nosql"],    [1.0, 0.0, 0.0, 0.0], "au1"),
    ("A2", "Aggregation pipeline pas cu pas", "ro", 2024, ["mongodb", "pipeline"], [0.6, 0.8, 0.0, 0.0], "au1"),
    ("A3", "Indecși B-tree și regula ESR",    "en", 2024, ["index", "mongodb"],    [0.5, 0.5, 0.5, 0.5], "au2"),
    ("A4", "Embeddings pentru text",          "ro", 2025, ["nlp", "vector"],       [0.0, 0.6, 0.8, 0.0], "au3"),
    ("A5", "HNSW explicat",                   "en", 2025, ["vector", "index"],     [0.0, 0.0, 0.6, 0.8], "au2"),
    ("A6", "Tokenizare BPE",                  "ro", 2025, ["nlp"],                 [0.0, 0.0, 0.0, 1.0], "au3"),
    ("A7", "Căutare hibridă",                 "ro", 2026, ["vector", "search"],    [0.0, 0.8, 0.0, 0.6], "au4"),
    ("A8", "Quantization pentru vectori",     "en", 2026, ["vector"],              [0.8, 0.0, 0.6, 0.0], "au4"),
]
AUTHORS = [
    {"_id": "au1", "name": "Ana Ionescu",    "country": "RO"},
    {"_id": "au2", "name": "John Miller",    "country": "US"},
    {"_id": "au3", "name": "Elena Dumitru",  "country": "RO"},
    {"_id": "au4", "name": "Lena Schmidt",   "country": "DE"},
]
q = [0.5, 0.7, 0.5, 0.1]

articles = lab["articles"]
authors = lab["authors"]
articles.delete_many({})
authors.delete_many({})
articles.insert_many([
    {"_id": _id, "title": title, "lang": lang, "year": year, "tags": tags,
     "text": f"{title}. Text demonstrativ pentru cursul BDNV.", "authorId": author,
     "embedding": embedding, "emb_model": "course-demo-4d", "emb_norm": True}
    for _id, title, lang, year, tags, embedding, author in ARTICLES
])
authors.insert_many(AUTHORS)

for document in articles.find({}, {"title": 1, "embedding": 1}):
    norm = math.sqrt(sum(x * x for x in document["embedding"]))
    print(f"{document['_id']}  ||e|| = {norm:.3f}  {document['title']}")
print(f"||q|| = {math.sqrt(sum(x * x for x in q)):.3f}")

### From brute force to a vector index

Before vector indexes are introduced, the similarity between the query and every document can be computed with ordinary aggregation expressions (Course 4, slide 17). For L2-normalized vectors, the **dot product equals the cosine similarity** (Course 3). `$reduce` iterates over the positions `0 .. d-1` produced by `$range` and accumulates the products of the corresponding components.

The function `dot_product` builds this expression for any field and query vector; it is reused below on the embeddings of real listing summaries. The cell shows the two documents of the handout's worked example (A3 and A2); the complete ranking is computed in Exercise 1.

In [ ]:
def dot_product(path: str, vector: list[float]) -> dict:
    # Aggregation expression for the dot product between the array field `path` and `vector`.
    return {"$reduce": {
        "input": {"$range": [0, {"$size": f"${path}"}]},
        "initialValue": 0,
        "in": {"$add": ["$$value", {"$multiply": [{"$arrayElemAt": [f"${path}", "$$this"]},
                                                  {"$arrayElemAt": [{"$literal": vector}, "$$this"]}]}]},
    }}


brute_force = [
    {"$set": {"dot": {"$round": [dot_product("embedding", q), 4]}}},
    {"$sort": {"dot": -1}},
    {"$limit": 2},
    {"$project": {"title": 1, "dot": 1}},
]
show(articles.aggregate(brute_force))

## Part I: Vector search

### Exact and approximate search

The brute-force pipeline above is an **exact nearest-neighbor (ENN)** search: it compares the query with every vector, so its recall is 100% and its cost is $O(n \cdot d)$. It is the right choice for small collections, for very selective filters, and as the **ground truth** against which an approximate search is measured. An **approximate nearest-neighbor (ANN)** search explores a graph of neighbors, has a cost that grows roughly logarithmically with $n$, and a recall below 100% that can be tuned (Course 4, slide 42).

MongoDB uses **HNSW** (*Hierarchical Navigable Small World*; Malkov & Yashunin, 2020) for ANN. The vectors are nodes of a graph organized in layers: the upper layers contain few nodes connected by long edges, and layer 0 contains all the vectors. A query starts at the top layer, moves greedily towards the query vector, and descends one layer at a time (Course 4, slide 43: highway, national road, street). The parameters are `maxEdges` (16–64 neighbors per node) and `numEdgeCandidates` (100–3200, the quality of the construction) at build time, and `numCandidates` (the width of the search) at query time.

### Architecture: `mongod` and `mongot`

Vector indexes are not B-trees and are not maintained by `mongod`. They are built by **`mongot`**, the search process based on Apache Lucene, which receives the changes of the collection through change streams (Course 4, slide 44). Consequently:

- a vector index is created with **`createSearchIndex()`** (in PyMongo, `create_search_index` with a `SearchIndexModel`), not with `createIndex()`;
- it is built **asynchronously** (`PENDING → BUILDING → READY`) and is **eventually consistent**: a newly inserted document appears in the results after a short delay;
- on Atlas, `mongot` runs on the cluster or on dedicated Search Nodes; since June 2026 it is also generally available for MongoDB Community Edition.

### The choice of the similarity function

| `similarity` | Measures | When | `vectorSearchScore` |
|---|---|---|---|
| `cosine` | the angle between the vectors | non-normalized vectors | $(1 + \cos) / 2$ |
| `dotProduct` | $a \cdot b$ (angle and norm) | L2-normalized vectors (fastest) | $(1 + a \cdot b) / 2$ |
| `euclidean` | the distance $\lVert a - b \rVert$ | models trained with the L2 distance | $1 / (1 + d)$ |

For $\lVert a \rVert = \lVert b \rVert = 1$, $\lVert a - b \rVert^2 = 2 - 2\cos(a, b)$: the three functions produce the same ranking, and `dotProduct` is the cheapest. The score is normalized to $[0, 1]$ (Course 4, slide 46).

### A vector index on the course dataset

The index `articles_vec` declares the vector field (`numDimensions` must equal the output dimension of the model) and the fields `lang` and `year` as **`filter`** fields, which can be used for pre-filtering. The cell first counts the search indexes that already exist, because of the limit of three on M0.

In [ ]:
existing = {c.full_name: [i["name"] for i in search_indexes(c)]
            for c in (airbnb, mflix["movies"], articles, lab["listings_emb"])}
print("Existing search indexes:", existing, f"(total: {sum(map(len, existing.values()))}; M0 allows 3)\n")

ARTICLES_INDEX = "articles_vec"
ensure_search_index(articles, ARTICLES_INDEX, {"fields": [
    {"type": "vector", "path": "embedding", "numDimensions": 4, "similarity": "dotProduct"},
    {"type": "filter", "path": "lang"},
    {"type": "filter", "path": "year"},
]})

### `$vectorSearch`

The syntax of the stage (Course 4, slide 47):

```python
{"$vectorSearch": {
    "index": "articles_vec",       # name of the vector index
    "path": "embedding",           # vector field
    "queryVector": q,              # embedded with the SAME model as the documents
    "numCandidates": 60,           # ANN: width of the search, >= 20 x limit (recommendation)
    "limit": 3,                    # k
    "filter": {"lang": "ro"},      # optional pre-filter, only on fields declared as "filter"
}}
```

- `$vectorSearch` must be the **first stage** of the pipeline;
- `limit ≤ numCandidates ≤ 10,000`;
- `exact: True` requests an ENN search (without `numCandidates`);
- the score is obtained with `{"$meta": "vectorSearchScore"}`;
- the filter supports `$eq`, `$ne`, `$gt(e)`, `$lt(e)`, `$in`, `$nin`, `$and`, `$or`, `$nor`, only on fields declared as `filter`.

The function `vector_search` builds such a pipeline; it returns the pipeline rather than the results, so that it can also be passed to `explain`.

In [ ]:
def vector_search(query_vector: list[float], *, index: str, path: str = "embedding", limit: int = 3,
                  num_candidates: int | None = None, exact: bool = False, filter: dict | None = None,
                  then: list[dict] = ()) -> list[dict]:
    # Build a pipeline: $vectorSearch, the score as a field, then the stages in `then`.
    stage = {"index": index, "path": path, "queryVector": query_vector, "limit": limit}
    if exact:
        stage["exact"] = True
    else:
        stage["numCandidates"] = num_candidates or 20 * limit
    if filter:
        stage["filter"] = filter
    return [{"$vectorSearch": stage}, {"$set": {"score": {"$meta": "vectorSearchScore"}}}, *then]


# ENN, k = 2: the worked example of the handout
enn = list(articles.aggregate(vector_search(q, index=ARTICLES_INDEX, limit=2, exact=True,
                                            then=[{"$project": {"title": 1, "embedding": 1, "score": 1}}])))
for document in enn:
    dot = sum(a * b for a, b in zip(q, document["embedding"]))
    print(f"{document['_id']}  score = {document['score']:.4f}   (1 + q.e) / 2 = {(1 + dot) / 2:.4f}   "
          f"q.e = {dot:.2f}   {document['title']}")

**Points to observe.** The scores reported by MongoDB are $(1 + q \cdot e)/2$: A3 has $q \cdot e = 0.90$ and score $0.95$, A2 has $0.86$ and $0.93$, exactly as computed by hand in the handout and by the brute-force pipeline above. Note that the documents were inserted *before* the index was created; a document inserted afterwards would become visible only once `mongot` has processed the change.

### Pre-filtering vs. post-filtering

A condition on the metadata can be applied in two places (Course 4, slide 48):

- **Pre-filter** (`filter` inside `$vectorSearch`): the search is performed only among the documents that satisfy the condition, and up to $k$ results are returned. Only fields declared as `filter` in the index can be used.
- **Post-filter** (`$match` after `$vectorSearch`): the search returns the $k$ nearest documents of the whole collection, and the `$match` removes some of them, so **fewer than $k$ results** may remain. A post-filter is needed for conditions that cannot be part of the index: a threshold on the score, a regular expression, a computed field.

*The two most similar articles from 2025 onwards:*

In [ ]:
recent = {"year": {"$gte": 2025}}
project = [{"$project": {"title": 1, "year": 1, "score": {"$round": ["$score", 4]}}}]

print("Pre-filter (filter in $vectorSearch):")
show(articles.aggregate(vector_search(q, index=ARTICLES_INDEX, limit=2, exact=True, filter=recent, then=project)))

print("\nPost-filter ($match after $vectorSearch):")
post = list(articles.aggregate(vector_search(q, index=ARTICLES_INDEX, limit=2, exact=True,
                                             then=[{"$match": recent}] + project)))
show(post) if post else print("(no results)")

**Points to observe.** The two nearest articles overall, A3 and A2, are both from 2024. The post-filter removes them and returns **no result at all**, whereas the pre-filter searches among the articles from 2025–2026 and returns two. With a post-filter, `limit` and `numCandidates` must be increased so that enough documents survive the `$match`; whenever the condition can be declared in the index, the pre-filter is preferable.

A `$match` placed after `$vectorSearch` cannot use a B-tree index such as `{lang: 1}`: `$vectorSearch` produces the stream itself from the vector index, and every subsequent stage operates on that stream, not on the collection.

### Vector search on real embeddings

The remaining examples use real text: the descriptions (`summary`) of the `sample_airbnb` listings. Their embeddings come from one of two sources, selected automatically:

| Source | Model | Dimension | Collection, field and index |
|---|---|---|---|
| **Lab 3** (if `vector_index` exists) | `nomic-embed-text` (Ollama) | 768 | `sample_airbnb.listingsAndReviews`, `summary_embedding_nomic_embed_text`, `vector_index` |
| **Built here** (otherwise) | `sentence-transformers/all-MiniLM-L6-v2` | 384 | `bdnv_lab.listings_emb`, `summary_embedding_all_minilm_l6_v2`, `listings_vec` |

When the embeddings are built here, the cell copies 2,000 listings with a non-empty summary into `bdnv_lab.listings_emb`, embeds the summaries on the CPU (about one to three minutes on Colab, plus the download of the model, about 90 MB), stores each vector with the name of the model, and creates the index `listings_vec` with `dotProduct` (the vectors are normalized) and the filter fields `address.market` and `property_type`. The cell is idempotent: if the embeddings and the index exist, it only reuses them.

The function `embed_query` embeds a question **with the same model** as the documents, which is the first rule of the course. With `all-MiniLM-L6-v2`, it runs locally; with `nomic-embed-text`, it requires the Ollama server of Lab 3 (`USE_OLLAMA = True`) and the prefix `search_query: `. Set `EMBEDDINGS_SOURCE = "minilm"` to build the embeddings here even if the index of Lab 3 exists.

In [ ]:
EMBEDDINGS_SOURCE = "auto"      # "auto", "lab3" (reuse Airbnb_Embeddings.ipynb) or "minilm" (build here)
USE_OLLAMA = False              # only for the Lab 3 embeddings: embed questions with Ollama
N_EMB = 2_000                   # listings embedded here (keep it small on M0)

lab3_ready = any(i["name"] == "vector_index" and i.get("queryable") for i in search_indexes(airbnb))

if EMBEDDINGS_SOURCE == "lab3" or (EMBEDDINGS_SOURCE == "auto" and lab3_ready):
    assert lab3_ready, "The index vector_index of Airbnb_Embeddings.ipynb does not exist: use 'minilm'."
    EMBED_MODEL, EMBED_FIELD, VECTOR_INDEX = ("nomic-embed-text",
                                              "summary_embedding_nomic_embed_text", "vector_index")
    emb_coll = airbnb

    def embed_query(text: str) -> list[float] | None:
        if not USE_OLLAMA:
            return None                     # no model available: the callers fall back to a stored vector
        from langchain_ollama import OllamaEmbeddings
        return OllamaEmbeddings(model=EMBED_MODEL).embed_query("search_query: " + text)
else:
    from sentence_transformers import SentenceTransformer

    EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    EMBED_FIELD, VECTOR_INDEX = "summary_embedding_all_minilm_l6_v2", "listings_vec"
    emb_coll = lab["listings_emb"]
    st_model = SentenceTransformer(EMBED_MODEL)

    def embed_query(text: str) -> list[float]:
        return st_model.encode(text, normalize_embeddings=True).tolist()

    if emb_coll.count_documents({EMBED_FIELD: {"$exists": True}}) < N_EMB:
        emb_coll.delete_many({})
        listings = list(airbnb.aggregate([
            {"$match": {"summary": {"$type": "string", "$ne": ""}}},
            {"$sort": {"_id": 1}},
            {"$limit": N_EMB},
            {"$project": {"name": 1, "summary": 1, "price": 1, "property_type": 1, "room_type": 1,
                          "address.market": 1, "host.host_name": 1,
                          "review_scores.review_scores_rating": 1}},
        ]))
        start = time.perf_counter()
        vectors = st_model.encode([d["summary"] for d in listings], batch_size=64,
                                  normalize_embeddings=True, show_progress_bar=True)
        print(f"{len(listings)} summaries embedded in {time.perf_counter() - start:.0f} s")
        for document, vector in zip(listings, vectors):
            document[EMBED_FIELD] = vector.tolist()
            document["emb_model"] = EMBED_MODEL
        emb_coll.insert_many(listings)

    ensure_search_index(emb_coll, VECTOR_INDEX, {"fields": [
        {"type": "vector", "path": EMBED_FIELD,
         "numDimensions": st_model.get_sentence_embedding_dimension(), "similarity": "dotProduct"},
        {"type": "filter", "path": "address.market"},
        {"type": "filter", "path": "property_type"},
    ]})

with_embedding = {EMBED_FIELD: {"$exists": True}}
print(f"\nModel: {EMBED_MODEL} | collection: {emb_coll.full_name} | index: {VECTOR_INDEX} | "
      f"documents with an embedding: {emb_coll.count_documents(with_embedding)}")

#### Exact k-NN on the real embeddings

The brute-force pipeline is applied to the real embeddings. In place of a question, the embedding of one listing is used as the query ("listings similar to this one"). Since it is not known in advance whether stored vectors are normalized, the cell first computes their norms with the same `$reduce` technique; for non-normalized vectors, the dot product would have to be divided by the norms to obtain the cosine.

In [ ]:
fields = {"name": 1, "summary": 1, "address.market": 1, EMBED_FIELD: 1}
reference = (emb_coll.find_one({**with_embedding, "address.market": "Porto"}, fields)
             or emb_coll.find_one(with_embedding, fields))
q_ref, REF_MARKET = reference[EMBED_FIELD], reference["address"]["market"]
print(f"Reference listing: {reference['_id']} {reference['name']!r} ({REF_MARKET}), d = {len(q_ref)}")

# Are the stored vectors normalized?
norms = list(emb_coll.aggregate([
    {"$match": with_embedding},
    {"$limit": 200},
    {"$project": {"_id": 0, "norm": {"$sqrt": {"$reduce": {
        "input": f"${EMBED_FIELD}", "initialValue": 0,
        "in": {"$add": ["$$value", {"$multiply": ["$$this", "$$this"]}]}}}}}},
    {"$group": {"_id": None, "min": {"$min": "$norm"}, "max": {"$max": "$norm"}}},
]))
print("Norms of 200 stored vectors:", {k: round(v, 4) for k, v in norms[0].items() if k != "_id"})

brute_force_real = [
    {"$match": with_embedding},
    {"$set": {"dot": dot_product(EMBED_FIELD, q_ref)}},
    {"$sort": {"dot": -1}},
    {"$limit": 5},
    {"$project": {"name": 1, "market": "$address.market", "dot": {"$round": ["$dot", 4]}}},
]
start = time.perf_counter()
show(emb_coll.aggregate(brute_force_real))
print(f"\nExact k-NN in the pipeline: {(time.perf_counter() - start) * 1000:.0f} ms "
      f"for {emb_coll.count_documents(with_embedding)} vectors x {len(q_ref)} dimensions")

**Points to observe.** The first result is the reference listing itself, with a dot product of 1 for normalized vectors. The computation requires $n \times d$ multiplications per query and reads every document (a `COLLSCAN`); for 1 million documents with $d = 384$, about 384 million multiplications per query. The vector index avoids this cost by examining only a small part of the collection.

#### The index and a "more like this" search

The definition of the vector index shows the number of dimensions, the similarity function and the filter fields. The search below returns the listings most similar to the reference listing, in the same market (pre-filter), excluding the listing itself (post-filter).

In [ ]:
vector_index = next(iter(emb_coll.list_search_indexes(VECTOR_INDEX)), None)
assert vector_index and vector_index.get("queryable"), "Run the embeddings cell first."
print(json.dumps(vector_index["latestDefinition"], indent=2))

vector_field = next(f for f in vector_index["latestDefinition"]["fields"] if f["type"] == "vector")
DIMENSIONS, SIMILARITY = vector_field["numDimensions"], vector_field["similarity"]

# "More like this": the listings most similar to the reference listing, in the same market
more_like_this = vector_search(
    q_ref, index=VECTOR_INDEX, path=EMBED_FIELD, limit=6, num_candidates=120,
    filter={"address.market": REF_MARKET},
    then=[{"$match": {"_id": {"$ne": reference["_id"]}}},                  # post-filter: not itself
          {"$project": {"name": 1, "price": {"$toDouble": "$price"}, "score": {"$round": ["$score", 4]},
                        "summary": {"$substrCP": [{"$ifNull": ["$summary", ""]}, 0, 80]}}}])
print(f"\nReference: {reference['name']}: {reference['summary'][:120]}...\n")
show(emb_coll.aggregate(more_like_this))

### Tuning `numCandidates`: recall@k and latency

`numCandidates` determines how many candidates the HNSW search accumulates before the best `limit` are returned. Small values give low latency and lower recall; large values give higher recall and higher latency (Course 4, slide 50). The value should be chosen by **measurement**:

$$\text{recall@}k = \frac{|\text{ANN}_k \cap \text{ENN}_k|}{k},$$

where the ENN result (`exact: True`) is the ground truth. The cell uses the embeddings of 20 random listings as queries and measures recall@10 and the median latency for several values of `numCandidates`. The recommendation is to choose the **smallest value that reaches the target recall** (e.g. 0.95), starting from about $20 \times k$.

In [ ]:
import matplotlib.pyplot as plt

K = 10
CANDIDATES = [10, 20, 50, 100, 200, 400, 800]
queries = [d[EMBED_FIELD] for d in emb_coll.aggregate([{"$match": with_embedding}, {"$sample": {"size": 20}},
                                                     {"$project": {EMBED_FIELD: 1}}])]


def result_ids(pipeline) -> list:
    return [d["_id"] for d in emb_coll.aggregate(pipeline + [{"$project": {"_id": 1}}])]


ground_truth = [result_ids(vector_search(v, index=VECTOR_INDEX, path=EMBED_FIELD, limit=K, exact=True))
                for v in queries]
rows = []
for nc in CANDIDATES:
    recalls, latencies = [], []
    for v, truth in zip(queries, ground_truth):
        start = time.perf_counter()
        ann = result_ids(vector_search(v, index=VECTOR_INDEX, path=EMBED_FIELD, limit=K, num_candidates=nc))
        latencies.append((time.perf_counter() - start) * 1000)
        recalls.append(len(set(ann) & set(truth)) / K)
    rows.append((nc, sum(recalls) / len(recalls), median(latencies)))
    print(f"numCandidates = {nc:>4}   recall@{K} = {rows[-1][1]:.3f}   median latency = {rows[-1][2]:6.1f} ms")

fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.plot([r[0] for r in rows], [r[1] for r in rows], marker="o", label=f"recall@{K}")
ax1.set_xscale("log"); ax1.set_xlabel("numCandidates"); ax1.set_ylabel(f"recall@{K}")
ax2 = ax1.twinx()
ax2.plot([r[0] for r in rows], [r[2] for r in rows], marker="s", color="tab:orange", label="latency (ms)")
ax2.set_ylabel("median latency (ms, incl. network)")
fig.legend(loc="lower right"); ax1.set_title(f"ANN vs. ENN ({EMBED_MODEL.split('/')[-1]})"); plt.tight_layout(); plt.show()

**Points to observe.** With a few thousand vectors, the HNSW graph is small, and recall typically approaches 1.0 already for moderate values of `numCandidates`; the latency is dominated by the network round trip. The trade-off becomes pronounced with millions of vectors, with filters (which make parts of the graph unusable) and with quantization, all of which require larger values. Exercise 2 repeats the measurement with a pre-filter.

### A retrieval pipeline for RAG

Slide 49 of the course combines Part I and Part III into the retrieval step of a RAG system: (1) semantic retrieval with a pre-filter, (2) a relevance threshold on the score (a post-filter), (3) enrichment, (4) a minimal projection, without the embedding, and (5) the formatting of the context that is sent to the LLM. In `sample_airbnb`, the host is **embedded** in the listing, so the enrichment step needs no `$lookup`: the data model has already performed the join.

`embed_query` embeds the question with the same model as the documents. If no query model is available (the embeddings of Lab 3 without Ollama), the embedding of the reference listing is used instead, so that the cell still runs.

In [ ]:
question = "a quiet flat close to the beach"
SCORE_THRESHOLD = 0.65      # depends on the model and on the similarity function: inspect the scores first
market = "Rio De Janeiro"

query_vector = embed_query(question)
if query_vector is None:
    print("No query model available: using the embedding of the reference listing instead.")
    query_vector, market = q_ref, REF_MARKET

rag_retrieval = vector_search(
    query_vector, index=VECTOR_INDEX, path=EMBED_FIELD, limit=15, num_candidates=300,
    filter={"address.market": market},                                       # (1) pre-filter
    then=[
        {"$match": {"score": {"$gte": SCORE_THRESHOLD}}},                    # (2) relevance threshold
        {"$project": {"_id": 0, "name": 1, "score": {"$round": ["$score", 3]},
                      "price": {"$toDouble": "$price"},
                      "host": "$host.host_name",                             # (3) embedded, no $lookup
                      "chunk": {"$substrCP": [{"$ifNull": ["$summary", ""]}, 0, 300]}}},  # (4)
        {"$limit": 5},
    ])
retrieved = list(emb_coll.aggregate(rag_retrieval))

context = "\n".join(f"[{i}] {d['name']} (price {d['price']}, host {d['host']}, score {d['score']}): {d['chunk']}"
                    for i, d in enumerate(retrieved, start=1))          # (5) context for the LLM
print(f"Question: {question!r} in {market}\n")
print(context or "No document above the threshold: lower SCORE_THRESHOLD or increase limit.")

### Quantization: memory vs. precision

The HNSW index is kept in the memory of the search nodes, so its size determines the hardware (Course 4, slides 38 and 51). A `float32` vector of dimension $d$ occupies $4d$ bytes; the index can store it **quantized**:

| `quantization` | Bytes per vector | Reduction | Remark |
|---|---|---|---|
| `"none"` (float32) | $4d$ | 1× | full fidelity |
| `"scalar"` (int8) | $d$ | 4× | small loss of recall; a good choice for large collections |
| `"binary"` (1 bit) | $d/8$ | 32× | `numDimensions` a multiple of 8; lower recall, compensated by a larger `numCandidates` and rescoring with the full-fidelity vectors |

Quantization is declared in the index definition (it is not created here, because of the limit of three search indexes on M0):

```python
{"type": "vector", "path": EMBED_FIELD, "numDimensions": 768,
 "similarity": "cosine", "quantization": "scalar"}
```

Alternatively, a model can produce already quantized vectors (`BinData` of subtype `int8` or `int1`), indexed with `quantization: "none"`.

In [ ]:
def vector_memory(n: int, d: int) -> dict[str, str]:
    # Memory of the vectors only (without the HNSW graph) for n vectors of dimension d.
    def human(b):
        return f"{b / 1e9:.2f} GB" if b >= 1e9 else f"{b / 1e6:.1f} MB"
    return {"float32": human(n * d * 4), "scalar (int8)": human(n * d), "binary (1 bit)": human(n * d / 8)}


print("1 M x 384 (all-MiniLM-L6-v2):", vector_memory(1_000_000, 384))
n_emb = emb_coll.count_documents(with_embedding)
print(f"{n_emb} x {DIMENSIONS} ({EMBED_MODEL}):", vector_memory(n_emb, DIMENSIONS))

### Hybrid search: `$rankFusion`

Semantic search finds synonyms and paraphrases; lexical (full-text) search finds **exact terms**: error codes (`E11000`), model names, identifiers. Such rare tokens are split into sub-word pieces by the tokenizer (Course 2), and their embedding does not preserve the exact match. **Hybrid search** combines both rankings (Course 4, slide 52). `$rankFusion` (MongoDB 8.0+) uses **Reciprocal Rank Fusion**: each document receives, from every input pipeline in which it appears,

$$\text{score}(d) = \sum_{p} w_p \cdot \frac{1}{60 + \text{rank}_p(d)},$$

with ranks starting at 1 and the constant 60 fixed. `$scoreFusion` (MongoDB 8.3+) combines normalized scores instead of ranks.

The function `rrf` implements the formula, so that the hand computations of the handout can be verified. The cell checks the worked example: A3 is first in the semantic ranking and second in the lexical one.

In [ ]:
def rrf(rankings: dict[str, list[str]], weights: dict[str, float] | None = None, k: int = 60) -> list[tuple]:
    # Reciprocal Rank Fusion of several rankings (ranks start at 1).
    scores: dict[str, float] = {}
    for name, ranking in rankings.items():
        w = (weights or {}).get(name, 1)
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0) + w / (k + rank)
    return sorted(scores.items(), key=lambda item: -item[1])


handout_rankings = {
    "semantic": ["A3", "A2", "A4", "A8", "A7"],     # from Exercise 3 of the handout
    "keyword":  ["A5", "A3", "A8", "A7"],           # $search "vector index"
}
fused = dict(rrf(handout_rankings))
print(f"A3: 1/61 + 1/62 = {fused['A3']:.6f}")
print(f"A2: 1/62        = {fused['A2']:.6f}")

In `$rankFusion`, every input pipeline must be a *selection* pipeline (only `$match`, `$search`, `$vectorSearch`, `$sample`, `$geoNear`, `$sort`, `$skip`, `$limit`) and a *ranked* one (it begins with `$search`, `$vectorSearch` or `$geoNear`, or contains an explicit `$sort`). The second pipeline therefore does not have to be a full-text search. The next cell fuses the semantic ranking with a ranking of the articles tagged `vector` by recency, which requires no additional search index. The fused score is read with `{"$meta": "score"}`.

In [ ]:
hybrid = [
    {"$rankFusion": {
        "input": {"pipelines": {
            "semantic": [{"$vectorSearch": {"index": ARTICLES_INDEX, "path": "embedding",
                                            "queryVector": q, "numCandidates": 60, "limit": 5}}],
            "recentVector": [{"$match": {"tags": "vector"}}, {"$sort": {"year": -1, "_id": 1}}, {"$limit": 5}],
        }},
        "combination": {"weights": {"semantic": 1, "recentVector": 1}},
        "scoreDetails": True,
    }},
    {"$project": {"title": 1, "year": 1, "score": {"$meta": "score"}, "details": {"$meta": "scoreDetails"}}},
    {"$limit": 5},
]
try:
    for document in articles.aggregate(hybrid):
        ranks = {d["inputPipelineName"]: d.get("rank") for d in document["details"]["details"]}
        print(f"{document['_id']}  {document['score']:.6f}  ranks={ranks}  {document['title']}")
except OperationFailure as error:
    print(f"$rankFusion is not available on this server: {error.details.get('errmsg', error)}")

**Optional: a true lexical pipeline.** With a free slot for a fourth search index (Flex or paid tiers, a local Atlas deployment, or after dropping another index), a full-text index on `title` and `tags` allows the hybrid search of the course to be reproduced. Set `CREATE_TEXT_INDEX = True`.

In [ ]:
CREATE_TEXT_INDEX = False        # M0: at most 3 search indexes per cluster

if CREATE_TEXT_INDEX:
    ensure_search_index(articles, "articles_text",
                        {"mappings": {"dynamic": False, "fields": {"title": {"type": "string"},
                                                                    "tags": {"type": "string"}}}},
                        kind="search")
    lexical_hybrid = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "semantic": [{"$vectorSearch": {"index": ARTICLES_INDEX, "path": "embedding",
                                                "queryVector": q, "numCandidates": 60, "limit": 5}}],
                "keyword": [{"$search": {"index": "articles_text",
                                         "text": {"query": "vector index", "path": ["title", "tags"]}}},
                            {"$limit": 5}],
            }},
            "combination": {"weights": {"semantic": 0.6, "keyword": 0.4}},
        }},
        {"$project": {"title": 1, "score": {"$meta": "score"}}},
    ]
    show(articles.aggregate(lexical_hybrid))

### Automated embedding

In all the examples above, the application computes the embeddings, both at ingestion and at query time, and is responsible for using the same model. With **automated embedding** (Course 4, slide 53), the index field of type `autoEmbed` names the model, and `mongot` generates the embeddings itself (with Voyage AI models) on insert and update, as well as for the text of the query:

```javascript
db.articles.createSearchIndex("articles_auto", "vectorSearch", {
  fields: [
    { type: "autoEmbed", modality: "text", path: "text", model: "voyage-4" },
    { type: "filter", path: "lang" } ] })

db.articles.aggregate([
  { $vectorSearch: { index: "articles_auto", path: "text",
      query: "cum indexez vectori?",          // text, not a vector
      numCandidates: 200, limit: 10 } } ])
```

The index guarantees that the same model is used for documents and queries (the tokenizer–model pair of Course 2 is fixed by the index), at the price of a cost per token and of additional latency at ingestion. This feature is recent (2026), requires Voyage AI API keys and is therefore not executed here; check the exact syntax in the current documentation before using it.

### Best practices (Course 4, slide 54)

| Topic | Practice |
|---|---|
| Same model | ingestion = query (version, pooling, normalization); store `emb_model` in the document |
| Changing the model | a new field (`embedding_v2`) and a new index, then switch |
| Similarity | normalized vectors → `dotProduct` |
| Filters | declared in the index; prefer pre-filtering |
| `numCandidates` | measure recall@k against `exact: True` |
| Projection | never return the `embedding` field |
| Memory | scalar quantization; the index is kept in RAM on the search nodes |
| Chunking | text longer than the model's window (256 tokens for `all-MiniLM-L6-v2`) is truncated |

### Exercises for Part I

#### Exercise 1: ENN, ANN and filters by hand (handout, Exercise 3) *(in the lab, ~15 min)*

*Objective: to compute vector search results by hand and to verify them with `$vectorSearch`.*

1. Compute the dot product $q \cdot e$ and the `vectorSearchScore` of all eight articles. Verify with the brute-force pipeline (without `$limit`) and with `vector_search(q, index=ARTICLES_INDEX, limit=8, exact=True)`.
2. What is the exact (ENN) result for `limit: 3`?
3. Pre-filter: the same query with `filter: {"lang": "ro"}` and `limit: 3`. Which documents are returned?
4. Post-filter: `$vectorSearch` without a filter and `limit: 3`, followed by `{"$match": {"lang": "ro"}}`. What is returned? Compare with (3) and explain the difference.
5. An ANN search with a too small `numCandidates` returns `[A3, A2, A8]`. Compute recall@3 with respect to (2). Which parameter must be changed?
6. Estimate, with `vector_memory`, the memory needed for 2 million vectors of dimension 768 in the variants float32, scalar and binary.
7. Why must `$vectorSearch` be the first stage, and why does `{"$match": {"lang": "ro"}}` placed after it not use an index `{lang: 1}`?

#### Exercise 2: Recall with filters, on real data *(homework)*

*Objective: to measure how filters affect the recall of an ANN search and the number of results.*

1. Repeat the recall@10 measurement with the pre-filter `{"address.market": m}` for a small market and a large market (count the documents per market with a `$group`). Do the curves differ? Explain why in terms of the HNSW graph.
2. Implement the same restriction as a **post-filter** (`$match` on `address.market` after `$vectorSearch`, without `filter`). For `limit = 10`, how often are fewer than 10 results returned? How large must `limit` and `numCandidates` be to obtain 10 results in at least 95% of the queries?
3. Ask three questions of your own with `embed_query` and the RAG pipeline (e.g. "family house with a garden", "cheap room near the metro"). Inspect the scores of the first 15 results without the threshold, then choose a value of `SCORE_THRESHOLD` and justify it. Why does a good threshold depend on the model?
4. *(Only with the indexes of Lab 3.)* On `sample_mflix.movies` (index `fullplot_embedding_nomic_embed_text_index`, filter fields `year` and `genres`), compare a pre-filter on `genres` with a post-filter on `genres`.

#### Exercise 3: RRF by hand (handout, Exercise 4) *(in the lab, ~10 min)*

*Objective: to understand how Reciprocal Rank Fusion combines rankings.*

The rankings are those of the cell `handout_rankings` (semantic: A3, A2, A4, A8, A7; keyword: A5, A3, A8, A7).

1. Compute by hand the RRF score (weights 1 and 1) of A8, A7, A5 and A4, and the final order of the six documents. Verify with `rrf(handout_rankings)`.
2. Recompute with the weights `semantic: 0.6, keyword: 0.4` (`rrf(handout_rankings, {"semantic": 0.6, "keyword": 0.4})`). Which documents change their position?
3. Why does a document that appears in both lists rise above documents that are first in only one list?
4. Give an example of a query for which lexical search is better than semantic search, and one for which it is worse.

## Part II: Back to the agent: `explain()` as a guardrail

In Lab 3, the tool `run_aggregate` executed any pipeline written by the model, with three protections: a read-only user, a bounded output and a time limit (`maxTimeMS`). None of them concerns the **cost** of the pipeline: a pipeline that scans the whole collection is executed, and `maxTimeMS` stops it only after the cost has been paid. Lab 4 (Part II) showed that the plan can be inspected **before** execution: `explain` with the verbosity `queryPlanner` does not execute the query.

The tool `run_aggregate_guarded` below applies three rules before executing a pipeline on `articles_big`:

1. `$out` and `$merge` are refused (as in Lab 3);
2. if the plan of the query layer contains a `COLLSCAN` on a large collection, the pipeline is **rejected**, and the message returned to the model lists the indexes of the collection, so that the model can rewrite the pipeline (self-correction, as in Lab 2);
3. otherwise, the pipeline is executed with a bounded output and a time limit.

The policy is deliberately strict: some legitimate analytical queries (e.g. a count over the whole collection) are rejected. Alternative policies, such as a threshold on the estimated number of documents examined, are the subject of Exercise 4.

**Recap from Lab 4.** `explain()` returns the **plan** chosen by the query optimizer. A `COLLSCAN` stage reads every document of the collection; an `IXSCAN` stage reads only a range of an index. A compound index `{lang: 1, year: -1}` can serve a query only through a **prefix** of its fields (`lang`, or `lang` and `year`), not through `year` alone.

The next cell generates the collection `articles_big` (100,000 synthetic articles, with the structure of `articles`) on the server, as in Part II of Lab 4, and creates two compound indexes. It is skipped if the collection already exists.

In [ ]:
N_BIG = 100_000            # keep this value on M0 (512 MB of storage)
N_AUTHORS = 1_000
LANGS = ["ro", "en", "fr", "de", "it"]
TAGS = ["mongodb", "nosql", "vector", "index", "nlp", "pipeline", "search", "rag"]
LOREM = ("Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt "
         "ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation.")


def random_int(low: int, high: int) -> dict:
    # Expression: a random integer in [low, high) (a 64-bit integer for large ranges).
    convert = "$toInt" if high - low < 2**31 else "$toLong"
    return {convert: {"$add": [low, {"$floor": {"$multiply": [{"$rand": {}}, high - low]}}]}}


def random_choice(values: list) -> dict:
    # Expression: a random element of `values`.
    return {"$arrayElemAt": [values, random_int(0, len(values))]}


articles_big = lab["articles_big"]
if articles_big.estimated_document_count() < N_BIG:
    articles_big.drop()
    lab.aggregate([
        {"$documents": [{"i": {"$range": [0, N_BIG]}}]},
        {"$unwind": "$i"},
        {"$project": {
            "_id": "$i",
            "title": {"$concat": ["Article ", {"$toString": "$i"}]},
            "lang": random_choice(LANGS),
            "year": random_int(2015, 2027),
            "tags": {"$setUnion": [[random_choice(TAGS), random_choice(TAGS)]]},
            "authorId": random_int(0, N_AUTHORS),
            "status": random_choice(["published", "published", "published", "draft"]),
            "text": {"$substrCP": [LOREM, 0, random_int(40, 180)]},
        }},
        {"$out": "articles_big"},
    ])

articles_big.create_index([("lang", 1), ("year", -1)])
articles_big.create_index([("lang", 1), ("year", -1), ("authorId", 1)], name="ESR")
print(f"articles_big: {articles_big.count_documents({})} documents, indexes: {list(articles_big.index_information())}")
print("\nPlan of a query on `year` only:")
_ = plan_summary(explain(articles_big, filter={"year": 2025}))

In [ ]:
from bson import json_util
from langchain.tools import tool

GUARD_MIN_DOCUMENTS = 10_000          # smaller collections are never rejected
FORBIDDEN_STAGES = {"$out", "$merge"}
MAX_DOCS, MAX_CHARS = 20, 6000


def parse_json(value, argument: str):
    # As in Lab 3: JSON object/array or JSON string (MongoDB Extended JSON) -> Python.
    try:
        return json_util.loads(value if isinstance(value, str) else json.dumps(value))
    except Exception as error:
        raise ValueError(f"`{argument}` is not valid JSON: {error}") from error


def check_pipeline(collection, pipeline: list[dict]) -> str | None:
    # Return None if the pipeline may be executed, otherwise the reason for rejecting it.
    stages = [next(iter(stage)) for stage in pipeline]
    if forbidden := FORBIDDEN_STAGES.intersection(stages):
        return f"Rejected: the stages {sorted(forbidden)} write to the database and are not allowed."
    if stages and stages[0] in ("$vectorSearch", "$search", "$geoNear"):
        return None
    if collection.estimated_document_count() < GUARD_MIN_DOCUMENTS:
        return None
    plan = plan_summary(explain(collection, pipeline, verbosity="queryPlanner"), quiet=True)
    if "COLLSCAN" in plan["plan"]:
        indexes = [dict(index["key"]) for index in collection.list_indexes()]
        return ("Rejected: this pipeline would read the whole collection (COLLSCAN). Begin the pipeline "
                "with a $match (and, if needed, a $sort) on the leading fields of one of these indexes: "
                f"{json.dumps(indexes)}")
    return None


@tool("run_aggregate_guarded")
def run_aggregate_guarded(pipeline: list | str) -> str:
    # The docstring below is the description that the model reads.
    """Run a read-only aggregation pipeline on the articles_big collection
    (fields: _id, title, lang, year, tags, authorId, status, text).
    The plan is checked with explain() first: a pipeline that would scan the whole
    collection is rejected, with the list of indexes it should use.
    Returns at most 20 documents as JSON, or an error or rejection message."""
    try:
        stages = parse_json(pipeline, "pipeline")
        if reason := check_pipeline(articles_big, stages):
            return reason
        documents = list(articles_big.aggregate(stages + [{"$limit": MAX_DOCS}], maxTimeMS=10_000))
        text = json.dumps(documents, default=str, ensure_ascii=False)
        return f"{len(documents)} documents: {text[:MAX_CHARS]}"
    except Exception as error:
        return f"Error: {error}"


tests = {
    "regex on text (no index)": [{"$match": {"text": {"$regex": "dolor"}}}, {"$count": "n"}],
    "filter on year only": [{"$match": {"year": 2025}}, {"$group": {"_id": "$lang", "n": {"$sum": 1}}}],
    "$out": [{"$match": {"lang": "ro"}}, {"$out": "stolen"}],
    "lang + year (indexed)": [{"$match": {"lang": "ro", "year": {"$gte": 2025}}},
                              {"$group": {"_id": "$year", "n": {"$sum": 1}}}, {"$sort": {"_id": 1}}],
}
for label, pipeline in tests.items():
    print(f"--- {label}\n{run_aggregate_guarded.invoke({'pipeline': pipeline})}\n")

**Points to observe.**

1. The first two pipelines are rejected before they are executed, and the rejection message is an observation that the model can act upon: it names the indexes, whose leading field is `lang`.
2. The second rejection is instructive: a filter on `year` alone does not use `{lang: 1, year: -1}` (the prefix rule of Part II). A model that does not know the prefix rule receives it in the tool result.
3. The guard uses `queryPlanner`, so the cost of the check is that of planning, not of execution.

**Optional: the guarded tool in an agent.** With the Ollama server and `qwen2.5:7b` of Lab 3, the tool can be given to an agent. Set `RUN_AGENT = True` and observe in the trace whether the model rewrites a rejected pipeline.

In [ ]:
RUN_AGENT = False        # requires the Ollama server and qwen2.5:7b (see Lab 3)

if RUN_AGENT:
    from langchain.agents import create_agent
    from langchain.chat_models import init_chat_model

    agent_llm = init_chat_model(model="qwen2.5:7b", model_provider="ollama", temperature=0)
    guard_prompt = """You are an expert MongoDB assistant for the collection articles_big
(fields: _id, title, lang, year, tags (array), authorId, status, text).
Answer the user's question by calling run_aggregate_guarded with an aggregation pipeline.
If the tool rejects the pipeline, rewrite it as the message suggests and call the tool again.
Base your answer only on the documents returned by the tool, and include the pipeline you used."""
    agent = create_agent(agent_llm, tools=[run_aggregate_guarded], system_prompt=guard_prompt)
    result = agent.invoke({"messages": [{"role": "user", "content":
        "How many articles were published in each year since 2024? And how many of them are in Romanian?"}]})
    for message in result["messages"]:
        message.pretty_print()

#### Exercise 4: An explain-aware tool for an agent *(homework)*

*Objective: to move a cost guarantee from the prompt into the harness.*

1. Adapt `check_pipeline` to a tool that runs pipelines on `sample_airbnb.listingsAndReviews` (as `run_aggregate` of Lab 3). Which indexes does the collection have (`list_indexes()`), and which typical questions about listings would be rejected?
2. Replace the rule "no `COLLSCAN`" by a threshold: execute `explain` with `executionStats` on the pipeline with a `$limit` appended and reject it if `totalDocsExamined` exceeds a fixed multiple of `nReturned`. Discuss the advantage of this policy and its cost (the check now executes part of the query).
3. With Ollama (Lab 3, or the optional cell above), run an agent with and without the guard on three questions of your choice, one of which filters on a non-indexed field. Compare the traces: how many tool calls are needed, and does the model follow the indications of the rejection message?
4. Which of the guarantees discussed so far (read-only user, `maxTimeMS`, bounded output, plan check) would you implement as middleware (Lab 3) rather than inside the tool? Justify the choice of hook (`wrap_tool_call`, `after_model`, ...).

## Cleaning up

The search indexes created here occupy slots of the three allowed on an M0 cluster, and the scratch database occupies storage. Set `CLEAN_UP = True` when you have finished the exercises. The sample databases and the indexes of Lab 3 are not affected.

In [ ]:
CLEAN_UP = False

if CLEAN_UP:
    for collection in (articles, lab["listings_emb"]):
        for index in search_indexes(collection):
            collection.drop_search_index(index["name"])
            print(f"Dropped search index {index['name']} on {collection.full_name}")
    client.drop_database(lab.name)
    print(f"Dropped database {lab.name}")

### Summary of the key concepts

| Concept | Role in this laboratory |
|---|---|
| Brute-force similarity | `$reduce` over the components: exact, $O(n \cdot d)$, a `COLLSCAN` |
| ENN vs. ANN | Exact search as ground truth; approximate search on an HNSW graph |
| `mongod` and `mongot` | Vector indexes built by `mongot` with `createSearchIndex()`, asynchronously, eventually consistent |
| Similarity function | `dotProduct` for normalized vectors; `vectorSearchScore` normalized to $[0, 1]$ |
| `$vectorSearch` | First stage; `numCandidates`, `limit`, `exact`, `filter` |
| Pre- vs. post-filter | Filter fields declared in the index vs. `$match` after the search (possibly fewer than k results) |
| Same model | Documents and queries embedded with the same model; `emb_model` stored in the document |
| recall@k | Tuning `numCandidates` against `exact: True` |
| Retrieval for RAG | Pre-filter, score threshold, minimal projection, context for the LLM |
| Quantization | float32 / int8 / 1 bit: memory vs. recall |
| `$rankFusion` (RRF) | Hybrid search: semantic and lexical (or other ranked) pipelines combined by rank |
| Automated embedding | `autoEmbed`: the index computes the embeddings of documents and queries |
| `explain()` as a guardrail | A tool that checks the plan of a model-written pipeline before executing it |

## References

**Official documentation**
- MongoDB. *Atlas Vector Search Overview*; *How to Index Fields for Vector Search*; *Run Vector Search Queries*. https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-overview/
- MongoDB. *Vector Quantization*. https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-quantization/
- MongoDB. *$rankFusion*. MongoDB Manual. https://www.mongodb.com/docs/manual/reference/operator/aggregation/rankfusion/
- MongoDB. *Atlas Search Limitations* and *Atlas Free Cluster Limits*. https://www.mongodb.com/docs/atlas/atlas-search/limitations/, https://www.mongodb.com/docs/atlas/reference/free-shared-limitations/
- MongoDB. *Explain Results*. MongoDB Manual. https://www.mongodb.com/docs/manual/reference/explain-results/
- PyMongo. *PyMongo Documentation* (`aggregate`, `create_search_index`, `SearchIndexModel`). https://pymongo.readthedocs.io/
- Sentence Transformers. *Documentation* and the model card of `sentence-transformers/all-MiniLM-L6-v2`. https://sbert.net, https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- LangChain. *Tools*; *Agents*. https://docs.langchain.com/oss/python/langchain/agents

**Course materials**
- *BDNV, Course 4: MongoDB: Aggregation Pipelines, Optimizare & Vector Search*, and the handout of Course 4.
- *BDNV, Lab 4: MongoDB Basics, Aggregation Pipelines and Query Optimization*.
- I. Banu. *Lab 2: Tools and Short-Term Memory*, *Lab 3: Middleware and Agent Customization*, and `Airbnb_Embeddings.ipynb`.

**Research papers**
- Malkov, Y. A., & Yashunin, D. A. (2020). *Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs*. IEEE Transactions on Pattern Analysis and Machine Intelligence, 42(4), 824–836. https://arxiv.org/abs/1603.09320
- Reimers, N., & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. Proceedings of EMNLP-IJCNLP. https://arxiv.org/abs/1908.10084
- Cormack, G. V., Clarke, C. L. A., & Büttcher, S. (2009). *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods*. Proceedings of the 32nd International ACM SIGIR Conference. https://doi.org/10.1145/1571941.1572114
- Lewis, P., Perez, E., Piktus, A., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. Advances in Neural Information Processing Systems (NeurIPS). https://arxiv.org/abs/2005.11401

> **Note on the preparation of this document.** Parts of this laboratory guide were drafted with the assistance of Claude (Anthropic), an AI assistant.